# Knowledge Graph using Neo4j

### Initialize Graph Store

To launch Neo4j locally, first ensure you have docker installed. Then, you can launch the database with the following docker command

```bash
docker run \
    -p 7474:7474 -p 7687:7687 \
    -v $PWD/data:/data -v $PWD/plugins:/plugins \
    --name neo4j-apoc \
    -e NEO4J_apoc_export_file_enabled=true \
    -e NEO4J_apoc_import_file_enabled=true \
    -e NEO4J_apoc_import_file_use__neo4j__config=true \
    -e NEO4JLABS_PLUGINS=\[\"apoc\"\] \
    neo4j:2026.02
```

From here, you can open the db at [http://localhost:7474/](http://localhost:7474/). On this page, you will be asked to sign in. Use the default username/password of `neo4j` and `neo4j`.

Once you login for the first time, you will be asked to change the password.

After this, you are ready to create your first property graph!

In [ ]:
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

KG_PERSIST_DIR = "./storage/property_graph"

graph_store = Neo4jPropertyGraphStore(
    username="neo4j",
    password=neo4j_pw,
    url="bolt://localhost:7687",
    database="neo4j",
)

### SCHEMA — derived from metadata fields

In [ ]:
ENTITIES = [
    "GEAR",
    "MATERIAL",
    "MODULE",
    "FAMILY",
    "ARTICLE",
]

RELATIONS = [
    "HAS_MATERIAL",
    "HAS_MODULE",
    "HAS_FAMILY",
    "HAS_ARTICLE_NR",
    "HAS_TORQUE",
    "HAS_TEETH_COUNT",
    "HAS_WEIGHT",
    "HAS_PITCH_CIRCLE_DIAMETER",
    "HAS_TIP_DIAMETER",
    "HAS_HUB_DIAMETER",
    "HAS_WIDTH",
    "HAS_LENGTH",
    "HAS_INNER_DIAMETER",
    "HAS_ANGLE_OF_ENGAGEMENT",
    "IS_STRAIGHT_TOOTHED",
    "HAS_VARIANT",
]

SCHEMA_VALIDATION_SCHEMA = {
    "GEAR":     RELATIONS,
    "MATERIAL": ["HAS_MATERIAL"],
    "MODULE":   ["HAS_MODULE"],
    "FAMILY":   ["HAS_FAMILY"],
    "ARTICLE":  ["HAS_ARTICLE_NR"],
}

### Build+Load KG

In [ ]:
# BUILD
def build_property_graph_index(
    all_nodes,
    persist_dir: str = KG_PERSIST_DIR,
    max_paths_per_chunk: int = 15,
):
    storage_context = StorageContext.from_defaults(graph_store=graph_store)

    kg_extractor = SchemaLLMPathExtractor(
        llm=Settings.llm,
        possible_entities=ENTITIES,
        possible_relations=RELATIONS,
        kg_validation_schema=SCHEMA_VALIDATION_SCHEMA,
        strict=True,
        num_workers=4,
        max_paths_per_chunk=max_paths_per_chunk,
    )

    index = PropertyGraphIndex(
        nodes=all_nodes,
        storage_context=storage_context,
        embed_model=Settings.embed_model,
        llm=Settings.llm,
        kg_extractors=[kg_extractor],
        show_progress=True,
    )

    Path(persist_dir).mkdir(parents=True, exist_ok=True)
    index.storage_context.persist(persist_dir=persist_dir)

    return index

# LOAD
def load_property_graph_index(
    persist_dir: str = KG_PERSIST_DIR,
):
    storage_context = StorageContext.from_defaults(
        persist_dir=persist_dir,
        graph_store=graph_store,
    )
    return load_index_from_storage(storage_context)

# GET OR BUILD
def get_or_build_property_graph_index(
    all_nodes,
    persist_dir: str = KG_PERSIST_DIR,
):
    if Path(persist_dir).exists() and any(Path(persist_dir).iterdir()):
        return load_property_graph_index(persist_dir=persist_dir)
    return build_property_graph_index(all_nodes=all_nodes, persist_dir=persist_dir)

### Build KG-Retriever

In [ ]:
def build_kg_retriever(
    property_graph_index: PropertyGraphIndex,
    similarity_top_k: int = 8,
    path_depth: int = 2,
    include_text: bool = True,
):
    vector_retriever = VectorContextRetriever(
        property_graph_store=property_graph_index.property_graph_store,
        embed_model=Settings.embed_model,
        similarity_top_k=similarity_top_k,
        path_depth=path_depth,
        include_text=include_text,
    )

    synonym_retriever = LLMSynonymRetriever(
        property_graph_store=property_graph_index.property_graph_store,
        llm=Settings.llm,
        path_depth=path_depth,
        include_text=include_text,
    )

    return property_graph_index.as_retriever(
        sub_retrievers=[vector_retriever, synonym_retriever],
        include_text=include_text,
    )

### Create it

In [ ]:
property_graph_index = get_or_build_property_graph_index(all_nodes=all_nodes)

kg_retriever = build_kg_retriever(
    property_graph_index=property_graph_index,
    similarity_top_k=8,
    path_depth=2,
    include_text=True,
)